# One-dimensional ($Q^2$-only) study: what does the $p_n$ slicing buy?

The production fit is two-dimensional, $\log_{10}Q^2 \times p_n$. The $p_n$ axis separates
quasi-elastic events (95 % of the selection below $p_n = 0.125$ GeV) from MEC and RES (40 to
50 % above 0.3 GeV) *inside* the fit: the high-$p_n$ bins pin the non-QE normalization while the
low-$p_n$ bins measure the axial form factor. A reviewer asked to see that the slices help. A
plain 1D-versus-2D comparison cannot show *why* they help, because two effects are entangled:
the in-situ QE/MEC separation and the better purity of the low-$p_n$ region. `xml/oned_study/`
(generated by `python/scripts/make_oned_study_xmls.py`) therefore removes the $p_n$ axis in two
ways, and this notebook lines them up with the `lowpn` / `lowpn_ctrl` variants of the $p_n$-range
study (notebook 22) and the full `count50` grid, from no $p_n$ information to the full grid:

| variant | reco grid | $p_n$ information | reads as |
|---|---|---|---|
| `q2_all` | 9 $Q^2$ bins, $p_n \in [0, 1]$ | none | the one-dimensional fit |
| `q2_lowpn` | 9 $Q^2$ bins, $p_n \in [0, 0.3]$ | a purity cut, no shape | what a CCQE-enrichment cut alone buys |
| `lowpn` | 9 $\times$ 4, $p_n < 0.3$ | slices below the cut | the slicing without the high-$p_n$ control region |
| `lowpn_ctrl` | 9 $\times$ 4 + one bin $[0.3, 1]$ | slices + a normalization control bin | the control-bin version of the argument |
| reference | 9 $\times$ 8 (`count50`) | full | the production grid |

Every variant keeps the production $\log_{10}Q^2$ edges and the widened spline box of the
binning study, so the natural reference is `binning_study_fit_results/<family>/count50`
(`REFERENCE_KIND = 'production'` compares with the production chains instead). The quantities
are those of notebooks 16 and 22: goodness of fit, $\delta_F(Q^2)$, the best-fit shift, the
fitted parameters and nuisance pulls, $r_A^2$ and the profile scans.

How to read the result:

* **Slicing versus purity (Asimov).** If `count50` beats `q2_all` but `q2_lowpn` is no better
  than `q2_all`, the gain comes from the in-situ separation, not from purity. If `q2_lowpn`
  already matches `count50`, a cut would do and the slices mainly buy the background control.
  `lowpn` and `lowpn_ctrl` show how much of the full-grid precision the slices below the cut
  carry on their own.
* **Degeneracy.** At 9 bins the fit cannot separate $M_A$ from the MEC and FSI knobs by shape,
  so the nuisance posteriors of `q2_all` should return their priors (width ratio near 1) and the
  profile parabola of $M_A$ should widen. $p_n$ does *not* break the $M_A$--RPA degeneracy (both
  are QE $Q^2$-shape effects), so no improvement is expected for that knob.
* **NuWro.** The best-fit shift of `q2_all` against the reference shows how much the 1D fit is
  pulled by mis-modelling it cannot see; a shift that the slices remove is the robustness
  argument.

Section 1 needs no fit: it is a linearized, statistics-only Fisher forecast of $\sigma(M_A)$ and
of the $M_A$--nuisance correlations per binning, which says what to expect and later serves as
a cross-check. Fits that have not finished are listed and skipped, so the notebook can be rerun
while the study is running.

In [ ]:
from pathlib import Path
import os
import re
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

# Locate ma_zexp whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(REPO / 'python' / 'scripts') not in sys.path:
    sys.path.insert(0, str(REPO / 'python' / 'scripts'))

from postfit_physical_parameters import PUBLICATION_RC, SPECS, _fa_curves, load_fit
import make_binning_study_xmls as binning
import make_lowpn_study_xmls as lowpn
import make_oned_study_xmls as oned

mpl.rcParams.update(PUBLICATION_RC)
SPEC = {spec.key: spec for spec in SPECS}

## Controls

`FAMILY` selects the fake data (`asimov` or `nuwro`; environment override `ONED_STUDY_FAMILY`).
`REFERENCE_KIND` picks the full-grid reference: `'binning'` is
`binning_study_fit_results/<family>/count50` (same widened spline box as the study XMLs),
`'production'` is `<family>_fit_results`. `INCLUDE_LOWPN` adds the `lowpn` and `lowpn_ctrl`
variants of the $p_n$-range study to the ladder when their fits exist. A reference fit that
predates the current nuisance set cannot be loaded (its chain lacks the new branches) and is
reported as missing. Leave `FIT_KEYS = None` to compare every fit with at least one finished
variant.

In [ ]:
OUTPUT_ROOT = Path(os.environ.get('AXIAL_MASS_OUTPUT_ROOT', '/nevis/hopper/data/epelaez/axial_mass'))
FAMILY = os.environ.get('ONED_STUDY_FAMILY', 'asimov')                # asimov or nuwro
REFERENCE_KIND = os.environ.get('ONED_STUDY_REFERENCE', 'binning')    # 'binning' (count50) or 'production'
INCLUDE_LOWPN = os.environ.get('ONED_STUDY_INCLUDE_LOWPN', '1') == '1'
STUDY_ROOTS = {'oned': OUTPUT_ROOT / 'oned_study_fit_results' / FAMILY,
               'lowpn': OUTPUT_ROOT / 'lowpn_study_fit_results' / FAMILY}
XML_ROOTS = {'oned': REPO / 'xml' / 'oned_study' / FAMILY, 'lowpn': REPO / 'xml' / 'lowpn_study' / FAMILY}
REFERENCE_ROOT = {'binning': OUTPUT_ROOT / 'binning_study_fit_results' / FAMILY / 'count50',
                  'production': OUTPUT_ROOT / f'{FAMILY}_fit_results'}[REFERENCE_KIND]
REFERENCE_LABEL = {'binning': 'count50 (binning study)', 'production': 'production'}[REFERENCE_KIND]
FIT_KEYS = None                      # e.g. ('ma', 'ma_uniform', 'minerva_k6', 'minerva_k6_nuisance')
Q2_REFERENCE = 0.50                  # GeV^2, the comparison point of notebook 6
Q2 = np.linspace(0.0, 2.0, 201)
BURN_IN, THIN = 0, 1
Q2_CHUNK = 25                        # Q^2 points evaluated at once: every chain sample is used
PROFILE_SCAN_FITS = ('ma', 'minerva_k6')   # fits whose 1D profile scans are overlaid in Section 8
INPUT_FILE = Path(binning.INPUT_FILE)
FIG_DIR = REPO / 'figs' / 'oned_study' / FAMILY
TABLE_DIR = REPO / 'tables' / 'oned_study'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PN_CUT = oned.PN_CUT
Q2_EDGES, PN_EDGES = np.asarray(oned.Q2_EDGES), np.asarray(oned.PN_EDGES)   # the full (count50) grid
REF = 'reference'

# The ladder, from no p_n information to the full grid: variant -> (study, log10(Q^2) edges, p_n edges)
LADDER = {name: ('oned', *oned.VARIANTS[name][:2]) for name in ('q2_all', 'q2_lowpn')}
if INCLUDE_LOWPN:
    LADDER |= {name: ('lowpn', *lowpn.VARIANTS[name][:2]) for name in ('lowpn', 'lowpn_ctrl')}
LADDER[REF] = (None, list(Q2_EDGES), list(PN_EDGES))
VARIANT_SOURCE = {v: src for v, (src, _, _) in LADDER.items()}
EDGES = {v: (tuple(q2), tuple(pn)) for v, (_, q2, pn) in LADDER.items()}
NBINS = {v: (len(q2) - 1) * (len(pn) - 1) for v, (q2, pn) in EDGES.items()}
VARIANT_COLOR = {'q2_all': '#D55E00', 'q2_lowpn': '#E69F00', 'lowpn': '#0072B2', 'lowpn_ctrl': '#56B4E9', REF: 'black'}
VARIANT_READS = {'q2_all': 'no p_n information', 'q2_lowpn': 'purity cut only', 'lowpn': 'slices below the cut',
                 'lowpn_ctrl': 'slices + control bin', REF: 'full grid'}

FIT_ORDER = [
    'ma', 'ma_uniform', 'minerva_k8', 'minerva_k7',
    'minerva_k6', 'lqcd_k7', 'lqcd_k6', 'minerva_lqcd_k7', 'minerva_lqcd_k6',
    'minerva_k6_nuisance', 'lqcd_k6_nuisance', 'minerva_lqcd_k6_nuisance',
    'minerva_k6_uniform', 'minerva_k6_uniform_nuisance',
]
# Fitted-parameter counts and uniform (penalty-free) parameters, as in notebooks 3, 16 and 22.
N_FITTED = {
    'ma': 6, 'ma_uniform': 6,
    'lqcd_k6': 2, 'lqcd_k7': 3, 'minerva_k6': 2, 'minerva_k7': 3,
    'minerva_k8': 4, 'minerva_lqcd_k6': 2, 'minerva_lqcd_k7': 3,
    'minerva_k6_uniform': 2, 'minerva_k6_nuisance': 7, 'lqcd_k6_nuisance': 7,
    'minerva_lqcd_k6_nuisance': 7, 'minerva_k6_uniform_nuisance': 7,
}
UNIFORM_PARAMETERS = {
    'ma_uniform': {'MACCQE'},
    'minerva_k6_uniform': {'FAzexpMinervaK6PCA1', 'FAzexpMinervaK6PCA2'},
    'minerva_k6_uniform_nuisance': {'FAzexpMinervaK6PCA1', 'FAzexpMinervaK6PCA2'},
}
print(f'family {FAMILY!r}: reference {REFERENCE_LABEL} at {REFERENCE_ROOT}')
for v, (src, q2, pn) in LADDER.items():
    print(f'  {v:12s} {len(q2) - 1:2d} x {len(pn) - 1:2d} = {NBINS[v]:3d} bins, p_n edges {binning.fmt_edges(pn)}'
          + (f'  <- {STUDY_ROOTS[src] / v}' if src else ''))

## 1. Statistics-only Fisher forecast per binning

Before any fit: the selected overlay MC (scaled to the fit's POT) gives the expected counts
$N_i$ per reco bin of each binning and, from the per-event knob vectors, the linear response
$\partial N_i/\partial\theta_a$ to one step of each fitted parameter: $M_A$ ($+0.1$ GeV, knot 4
over knot 3 of `MaCCQE_UBGenie`) and the five floated nuisances of the `ma` fit in the units of
their pulls (one knot step for the 7-point knobs, the alternative universe for the two-point
ones). The Fisher matrix $F_{ab} = \sum_i \partial_a N_i\,\partial_b N_i / N_i$ then gives
$\sigma(M_A)$ with the nuisances fixed, with unit Gaussian priors on their pulls, and free, and
the pairwise correlation $\rho(M_A, \theta_a) = F_{0a}/\sqrt{F_{00}F_{aa}}$, whose variance
inflation $1/(1-\rho^2)$ is the price of floating that knob alone. This is linear and ignores
the flux, cross-section and detector covariance, so the absolute numbers are optimistic; the
ratios between binnings are the forecast, and the correlations say which knobs the
one-dimensional fit can no longer separate from $M_A$ by shape ($\rho(M_A, \mathrm{RPA})$ should
not depend on the binning).

In [ ]:
GENIE_MODES = {1: 'QE', 10: 'MEC', 4: 'RES', 3: 'DIS', 5: 'COH'}
# label -> (branch, index of the +1 step, index of the CV); the order of the fitted pulls of the ma fit
RESPONSE_KNOBS = {
    'M_A': ('MaCCQE_UBGenie', 4, 3),                  # knot 3 = tuned CV, knot 4 = +0.1 GeV
    'RPA_CCQE': ('RPA_CCQE_UBGenie', 4, 3),           # knobvals -3.6 .. 2.4, one knot = one sigma
    'XSecShape_CCMEC': ('XSecShape_CCMEC_UBGenie', 1, 0),   # knobvals "0, 1"
    'MFP_N': ('MFP_N_UBGenie', 4, 3),
    'DecayAngMEC': ('DecayAngMEC_UBGenie', 0, 1),     # knobvals "1, 0"
    'FrCEx_N': ('FrCEx_N_UBGenie', 4, 3),
}
MA_STEP_GEV = 0.1
NUISANCES = [k for k in RESPONSE_KNOBS if k != 'M_A']


def pot_scale():
    # expected data counts = MC weights x (detector POT / MC POT), read from one study XML
    xml = next(iter(sorted((XML_ROOTS['oned'] / 'q2_all').glob('*.xml')))).read_text()
    detector = float(re.search(r'<detector[^>]*pot="([^"]+)"', xml).group(1))
    mc = float(re.search(r'<MCFile[^>]*pot="([^"]+)"', xml).group(1))
    return detector / mc


def load_response_arrays():
    import awkward as ak
    tree = uproot.open(INPUT_FILE)['tree']
    branches = [b for b, _, _ in RESPONSE_KNOBS.values() if b in tree.keys()]
    arrays = tree.arrays(['isdata', 'isext', 'isdirt', 'isnuwro', 'afro_1mu1p_sel', 'afro_1mu1p_Q2', 'afro_1mu1p_Pn',
                          'net_weight', 'wc_truth_nuScatType', *branches])
    overlay = ((arrays['isdata'] == 0) & (arrays['isext'] == 0) & (arrays['isdirt'] == 0)
               & (arrays['isnuwro'] == 0) & (arrays['afro_1mu1p_sel'] == 1))
    arrays = arrays[overlay]
    x = np.log10(ak.to_numpy(arrays['afro_1mu1p_Q2']).astype(float))
    y = ak.to_numpy(arrays['afro_1mu1p_Pn']).astype(float)
    w = ak.to_numpy(arrays['net_weight']).astype(float) * pot_scale()
    mode = ak.to_numpy(arrays['wc_truth_nuScatType'])
    response = {}
    for label, (branch, i_plus, i_cv) in RESPONSE_KNOBS.items():
        if branch not in branches:
            print(f'{label}: branch {branch} missing, skipped')
            continue
        # The knob vectors are absolute tuned weights (the CV knot carries the tune weight) and net_weight is
        # proportional to that CV knot, so w * (vec_plus / vec_cv - 1) stays finite even for the events the tune
        # suppressed to ~1e-29 that an alternative universe restores; PROfit's force_0_cv sees the same ratio.
        vec = ak.to_numpy(ak.fill_none(ak.pad_none(arrays[branch], max(i_plus, i_cv) + 1), 1.0)).astype(float)
        with np.errstate(divide='ignore', invalid='ignore'):
            r = vec[:, i_plus] / vec[:, i_cv] - 1
        bad = ~np.isfinite(r)
        if bad.any():
            print(f'{label}: {bad.sum()} event(s) with an invalid ratio set to no response')
        r = np.where(bad, 0.0, r)
        large = np.abs(w * r) > 1
        if large.any():
            print(f'{label}: {large.sum()} event(s) change the expected count by more than one event each '
                  f'({(w * r)[large].sum():+.1f} events in total)')
        response[label] = r
    return x, y, w, mode, response


def fisher(x, y, w, response, q2_edges, pn_edges):
    # expected counts and the Fisher matrix of (M_A, nuisances...) in pull units; empty bins are dropped
    n = np.histogram2d(x, y, bins=[q2_edges, pn_edges], weights=w)[0].ravel()
    J = np.stack([np.histogram2d(x, y, bins=[q2_edges, pn_edges], weights=w * r)[0].ravel() for r in response.values()], axis=1)
    keep = n > 0
    return (J[keep].T / n[keep]) @ J[keep], n


X, Y, W, MODE, RESPONSE = load_response_arrays()
LABELS = list(RESPONSE_KNOBS)
rows = []
for v, (q2, pn) in EDGES.items():
    F, n = fisher(X, Y, W, RESPONSE, q2, pn)
    prior = np.diag([0.0] + [1.0] * (len(LABELS) - 1))          # unit Gaussian on every nuisance pull, none on M_A
    sigma = {'fixed': 1 / np.sqrt(F[0, 0]), 'with priors': np.sqrt(np.linalg.inv(F + prior)[0, 0]),
             'free': np.sqrt(np.linalg.inv(F)[0, 0])}
    rho = F[0, 1:] / np.sqrt(F[0, 0] * np.diag(F)[1:])
    rows.append(dict(variant=v, bins=int((n > 0).sum()), events=n.sum(),
                     **{f'sigma(M_A) {k} [GeV]': MA_STEP_GEV * s for k, s in sigma.items()},
                     **{f'rho {k}': r for k, r in zip(LABELS[1:], rho)}))
FISHER = pd.DataFrame(rows).set_index('variant')
sigma_cols = [c for c in FISHER.columns if c.startswith('sigma')]
rho_cols = [c for c in FISHER.columns if c.startswith('rho')]
for c in sigma_cols:
    FISHER[c.replace('[GeV]', '/ ref')] = FISHER[c] / FISHER.loc[REF, c]
FISHER['inflation (worst knob)'] = 1 / (1 - FISHER[rho_cols].abs().max(axis=1) ** 2)
display(FISHER.style.format({c: '{:.4f}' for c in sigma_cols} | {c: '{:+.2f}' for c in rho_cols}
                            | {c: '{:.2f}' for c in FISHER.columns if '/ ref' in c or 'inflation' in c} | {'events': '{:.0f}'}))
FISHER.to_csv(TABLE_DIR / 'fisher_forecast.csv', float_format='%.5g')

fig, (ax_sigma, ax_rho) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={'width_ratios': [1.1, 1]})
variants = list(EDGES)
xpos = np.arange(len(variants))
for k, (case, hatch) in enumerate([('fixed', ''), ('with priors', '//'), ('free', 'xx')]):
    ax_sigma.bar(xpos + (k - 1) * 0.27, FISHER[f'sigma(M_A) {case} [GeV]'], width=0.26,
                 color=[VARIANT_COLOR[v] for v in variants], alpha=0.5 + 0.25 * k, hatch=hatch, edgecolor='black', lw=0.5,
                 label=f'nuisances {case}')
ax_sigma.set_xticks(xpos, [f'{v}\n({NBINS[v]} bins)' for v in variants], fontsize=8)
ax_sigma.set_ylabel(r'stat-only $\sigma(M_A)$ forecast [GeV]'); ax_sigma.legend(fontsize=8); ax_sigma.grid(axis='y', alpha=0.3)
image = ax_rho.imshow(FISHER[rho_cols].abs().T.to_numpy(), vmin=0, vmax=1, cmap='Reds', aspect='auto')
ax_rho.set_xticks(xpos, variants, fontsize=8); ax_rho.set_yticks(range(len(rho_cols)), [c.removeprefix('rho ') for c in rho_cols], fontsize=8)
for i, v in enumerate(variants):
    for j, c in enumerate(rho_cols):
        ax_rho.text(i, j, f'{FISHER.loc[v, c]:+.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(FISHER.loc[v, c]) > 0.6 else 'black')
fig.colorbar(image, ax=ax_rho, label=r'$|\rho(M_A, \theta)|$ (Fisher, stat only)')
fig.suptitle('linearized statistics-only forecast per binning (no covariance systematics)', fontsize=10)
fig.tight_layout(); fig.savefig(FIG_DIR.parent / 'fisher_forecast.pdf'); plt.show()

The composition of the $Q^2$ bins for the two one-dimensional variants, and their response to
one $M_A$ step and one MEC-shape step: in `q2_all` the MEC response has roughly the same $Q^2$
shape as the $M_A$ response, which is the degeneracy the slices remove.

In [ ]:
def composition(q2_edges, pn_lo, pn_hi):
    in_range = (Y >= pn_lo) & (Y < pn_hi)
    rows = []
    for lo, hi in zip(q2_edges[:-1], q2_edges[1:]):
        in_bin = in_range & (X >= lo) & (X < hi); total = W[in_bin].sum()
        row = {'log10Q2 bin': f'[{lo:g}, {hi:g}]', 'expected events': total}
        for code_, name in GENIE_MODES.items():
            row[f'{name} [%]'] = 100 * W[in_bin & (MODE == code_)].sum() / total
        for label in ('M_A', 'XSecShape_CCMEC', 'RPA_CCQE'):
            row[f'dN/N {label} [%]'] = 100 * (W[in_bin] * RESPONSE[label][in_bin]).sum() / total
        rows.append(row)
    return pd.DataFrame(rows)


ONED = {v: composition(EDGES[v][0], EDGES[v][1][0], EDGES[v][1][-1]) for v in ('q2_all', 'q2_lowpn')}
for v, table in ONED.items():
    print(v)
    display(table.style.format({c: '{:.1f}' for c in table.columns if '[%]' in c} | {'expected events': '{:.0f}'}))
    table.to_csv(TABLE_DIR / f'q2_composition_{v}.csv', index=False, float_format='%.4g')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for v, ls in zip(ONED, ['-', '--']):
    table, edges = ONED[v], np.asarray(EDGES[v][0])
    for name, color in zip(['QE', 'MEC', 'RES'], ['#56B4E9', '#E69F00', '#009E73']):
        axes[0].stairs(table[f'{name} [%]'], edges, baseline=None, color=color, ls=ls, lw=1.6, label=f'{name} ({v})')
    for label, color in zip(['M_A', 'XSecShape_CCMEC', 'RPA_CCQE'], ['black', '#E69F00', '#CC79A7']):
        axes[1].stairs(table[f'dN/N {label} [%]'], edges, baseline=None, color=color, ls=ls, lw=1.6, label=f'{label} ({v})')
axes[0].set_ylabel('fraction of the bin [%]'); axes[0].set_ylim(0, 100)
axes[1].axhline(0, color='grey', lw=0.8); axes[1].set_ylabel('relative change for one step [%]')
for ax in axes:
    ax.set_xlabel(r'reco $\log_{10}(Q^2 / \mathrm{GeV}^2)$'); ax.legend(fontsize=7, ncol=2)
fig.tight_layout(); fig.savefig(FIG_DIR.parent / 'q2_composition.pdf'); plt.show()

## 2. Inventory

A fit counts as finished when its `*_v1_PROfile.root` exists. The reference column shows which
fits can be loaded from the reference tree; `lowpn` and `lowpn_ctrl` come from the $p_n$-range
study tree when `INCLUDE_LOWPN` is set.

In [ ]:
def fit_dir(variant, fit):
    return REFERENCE_ROOT / fit if variant == REF else STUDY_ROOTS[VARIANT_SOURCE[variant]] / variant / fit


def profile_root(variant, fit):
    matches = sorted(fit_dir(variant, fit).glob('*_v1_PROfile.root'))
    return matches[0] if len(matches) == 1 else None


VARIANTS = [v for v in LADDER if v != REF and (XML_ROOTS[VARIANT_SOURCE[v]] / v).is_dir()]
candidate_fits = FIT_ORDER if FIT_KEYS is None else list(FIT_KEYS)
FINISHED = {v: [fit for fit in candidate_fits if profile_root(v, fit) is not None] for v in [REF, *VARIANTS]}
FITS = [fit for fit in candidate_fits if any(fit in FINISHED[v] for v in VARIANTS)]
if not FITS:
    raise RuntimeError(f'no finished fit below {STUDY_ROOTS["oned"]}; run xml/oned_study/run_all.sh --family {FAMILY}')
VARIANTS = [v for v in VARIANTS if FINISHED[v]]
INVENTORY = pd.DataFrame({v: {fit: ('done' if fit in FINISHED[v] else '') for fit in FITS} for v in [REF, *VARIANTS]})
INVENTORY.loc['bins'] = {v: NBINS[v] for v in [REF, *VARIANTS]}
INVENTORY.loc['reads as'] = {v: VARIANT_READS[v] for v in [REF, *VARIANTS]}
display(INVENTORY)
VARIANT_LABEL = {v: f'{v} ({NBINS[v]} bins)' for v in VARIANTS} | {REF: f'{REFERENCE_LABEL} ({NBINS[REF]} bins)'}

## 3. Goodness of fit

Same bookkeeping as notebooks 3 and 22: the pull penalty of the Gaussian-constrained parameters
is removed from PROfit's total, $N_{\rm dof}=N_{\rm bins}-N_{\rm fitted}$. With 9 bins and up to
7 fitted parameters the one-dimensional $N_{\rm dof}$ is small, so $\chi^2/N_{\rm dof}$ is noisy
there. On NuWro fake data a smaller penalty in `q2_all` than in the reference means the 1D fit
absorbs the mis-modelling into the axial parameter instead of pulling the nuisances; on Asimov
fake data the data term is zero by construction and only the penalty carries information.

In [ ]:
CHI2_PATTERN = re.compile(r'Global Best Fit chi(?:\^?2|²):\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)')
NBINS_PATTERN = re.compile(r'/\s*(\d+)\s+reco bins')
PARAM_PATTERN = re.compile(r'\|\|\s+(\w+)\s+:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)')


def read_fit_stats(log_file):
    if not log_file.is_file():
        return np.nan, np.nan, {}
    text = log_file.read_text(errors='replace')
    chi2 = CHI2_PATTERN.findall(text)
    nbins = NBINS_PATTERN.findall(text)
    final_block = text.rsplit('Global Best Fit chi^2:', 1)[-1]
    parameters = {name: float(value) for name, value in PARAM_PATTERN.findall(final_block)}
    return (float(chi2[-1]) if chi2 else np.nan), (int(nbins[-1]) if nbins else np.nan), parameters


records = []
for v in [REF, *VARIANTS]:
    for fit in FITS:
        if fit not in FINISHED[v]:
            continue
        chi2_tot, nbins_log, parameters = read_fit_stats(fit_dir(v, fit) / 'profile.log')
        penalty = sum(value ** 2 for name, value in parameters.items() if name not in UNIFORM_PARAMETERS.get(fit, set()))
        nbins = nbins_log if np.isfinite(nbins_log) else NBINS[v]
        ndof = nbins - N_FITTED[fit]
        records.append(dict(variant=v, fit=fit, nbins=nbins, ndof=ndof, chi2_tot=chi2_tot, chi2_penalty=penalty,
                            chi2_data=chi2_tot - penalty, chi2_data_per_ndof=(chi2_tot - penalty) / ndof))
CHI2 = pd.DataFrame(records)
chi2_table = CHI2.pivot(index='fit', columns='variant', values=['chi2_data', 'chi2_data_per_ndof', 'chi2_penalty'])
chi2_table = chi2_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['chi2_data', 'chi2_data_per_ndof', 'chi2_penalty'], [REF, *VARIANTS]]))
display(chi2_table.style.format('{:.3f}', na_rep=''))

CHI2_PANELS = [('chi2_data', r'$\chi^2_{\rm data}$'), ('chi2_data_per_ndof', r'$\chi^2_{\rm data}/N_{\rm dof}$'),
               ('chi2_penalty', r'$\chi^2_{\rm pull}$ (penalty of the constrained parameters)')]
order = [REF, *VARIANTS]
fig, axes = plt.subplots(1, len(CHI2_PANELS), figsize=(4.8 * len(CHI2_PANELS), 0.32 * len(FITS) + 2.2), sharey=True)
y = np.arange(len(FITS))[::-1]
for k, v in enumerate(order):
    part = CHI2[CHI2.variant == v].set_index('fit').reindex(FITS)
    offset = (k - (len(order) - 1) / 2) * 0.15
    for ax, (column, _) in zip(axes, CHI2_PANELS):
        ax.plot(part[column], y + offset, 'o', ms=5, color=VARIANT_COLOR[v], label=VARIANT_LABEL[v] if ax is axes[0] else None)
for ax, (column, label) in zip(axes, CHI2_PANELS):
    ax.set_xlabel(label); ax.grid(axis='x', alpha=0.3); ax.set_axisbelow(True)
axes[1].axvline(1, color='grey', lw=0.8)
axes[0].set_yticks(y, FITS, fontsize=8); axes[0].legend(fontsize=8)
fig.suptitle(f'{FAMILY}: goodness of fit along the p_n-information ladder against {REFERENCE_LABEL}', fontsize=10)
fig.tight_layout(); fig.savefig(FIG_DIR / 'chi2.pdf'); plt.show()

## 4. Posterior chains

`load_fit` transforms every chain to physical parameters and, for the fits with nuisances,
keeps the nuisance pulls next to them. A reference fit whose chain predates the current
nuisance set raises a `KeyError` on the missing branch and is skipped with a message.

In [ ]:
RESULTS = {v: {} for v in [REF, *VARIANTS]}
for v in [REF, *VARIANTS]:
    for fit in FITS:
        if fit not in FINISHED[v]:
            continue
        # load_fit reads <data_root>/<suite>/<fit>/: the reference tree, or the study tree of the variant
        root, suite = (REFERENCE_ROOT.parent, REFERENCE_ROOT.name) if v == REF else (STUDY_ROOTS[VARIANT_SOURCE[v]], v)
        try:
            result = load_fit(SPEC[fit], suite=suite, burn_in=BURN_IN, thin=THIN, data_root=root)
        except KeyError as error:
            print(f'{v:12s} {fit:28s} skipped: chain lacks {error} (fit predates the current nuisance set?)')
            continue
        if result is None:
            continue
        RESULTS[v][fit] = result
        print(f'{v:12s} {fit:28s} {len(result["samples"]):>8,} samples')
for fit in FITS:
    lengths = {v: len(RESULTS[v][fit]['samples']) for v in [REF, *VARIANTS] if fit in RESULTS[v]}
    if len(set(lengths.values())) > 1:
        print(f'NOTE {fit}: chain lengths differ, ' + ', '.join(f'{v}={n:,}' for v, n in lengths.items())
              + ' (widths and medians stay comparable, the shorter chains carry more MCMC noise)')

## 5. $F_A(Q^2)$: precision along the ladder

$\delta_F(Q^2)=\sigma[F_A]/|\mathbb{E}[F_A]|$ over every chain sample (definition of notebook 6).
The headline is $\delta_F^{\rm variant}/\delta_F^{\rm ref}$ at $Q^2_{\rm ref}$ along the ladder,
one line per fit: a step down from `q2_all` to `q2_lowpn` is what purity buys, from `q2_lowpn` to
`lowpn` what the slices buy on the same events, from `lowpn` to the reference what the high-$p_n$
control region buys. The mean shift is
$(\mathbb{E}[F_A]_{\rm variant}-\mathbb{E}[F_A]_{\rm ref})/\sigma[F_A]_{\rm ref}$. The per-fit
figures show $\delta_F(Q^2)$ and the posterior $-F_A$ band.

In [ ]:
def fa_band(result, q2, use_prior=False, quantiles=(16, 50, 84)):
    q2 = np.atleast_1d(np.asarray(q2, dtype=float))
    n = len(result['prior_samples' if use_prior else 'samples'])
    mean, sigma = np.empty(len(q2)), np.empty(len(q2))
    pct = np.empty((len(quantiles), len(q2)))
    for start in range(0, len(q2), Q2_CHUNK):
        stop = min(start + Q2_CHUNK, len(q2))
        points = q2[start:stop] if stop - start > 1 else np.r_[q2[start:stop], q2[start:stop] + 1e-6]
        curves = _fa_curves(result, points, use_prior=use_prior, max_samples=n)[:, :stop - start]
        mean[start:stop], sigma[start:stop] = curves.mean(axis=0), curves.std(axis=0)
        pct[:, start:stop] = np.percentile(curves, quantiles, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        frac = sigma / np.abs(mean)
    return dict(mean=mean, sigma=sigma, frac=frac, q16=pct[0], median=pct[1], q84=pct[2])


rows = []
BANDS = {v: {} for v in [REF, *VARIANTS]}
IREF = int(np.argmin(np.abs(Q2 - Q2_REFERENCE)))
IN_RANGE = (Q2 > 0.05) & (Q2 <= 1.0)
for fit in FITS:
    for v in [REF, *VARIANTS]:
        if fit in RESULTS[v]:
            BANDS[v][fit] = fa_band(RESULTS[v][fit], Q2)
    ref = BANDS[REF].get(fit)
    if ref is not None:
        rows.append(dict(fit=fit, variant=REF, nbins=NBINS[REF], **{'posterior [%]': 100 * ref['frac'][IREF],
                         'ratio to reference': 1.0, 'mean shift [sigma_ref]': 0.0, 'max ratio Q2<=1': 1.0}))
    for v in VARIANTS:
        band = BANDS[v].get(fit)
        if band is None:
            continue
        rows.append(dict(
            fit=fit, variant=v, nbins=NBINS[v],
            **{'posterior [%]': 100 * band['frac'][IREF],
               'ratio to reference': band['frac'][IREF] / ref['frac'][IREF] if ref is not None else np.nan,
               'mean shift [sigma_ref]': (band['mean'][IREF] - ref['mean'][IREF]) / ref['sigma'][IREF] if ref is not None else np.nan,
               'max ratio Q2<=1': np.nanmax(band['frac'][IN_RANGE] / ref['frac'][IN_RANGE]) if ref is not None else np.nan}))
FA = pd.DataFrame(rows)
fa_table = FA.pivot(index='fit', columns='variant', values=['posterior [%]', 'ratio to reference', 'mean shift [sigma_ref]'])
fa_table = fa_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['posterior [%]', 'ratio to reference', 'mean shift [sigma_ref]'], [REF, *VARIANTS]]))
print(f'delta_F at Q^2 = {Q2_REFERENCE} GeV^2, reference = {REFERENCE_LABEL}')
display(fa_table.style.format('{:.3f}', na_rep=''))

# Headline: the ladder, one line per fit (left: precision ratio, right: mean shift).
ladder = [*VARIANTS, REF]
xpos = np.arange(len(ladder))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
cmap = plt.get_cmap('tab20')
for k, fit in enumerate(FITS):
    part = FA[FA.fit == fit].set_index('variant').reindex(ladder)
    style = dict(color=cmap(k % 20), marker='o', ms=4, lw=1.2, label=fit)
    axes[0].plot(xpos, part['ratio to reference'], **style)
    axes[1].plot(xpos, part['mean shift [sigma_ref]'], **style)
axes[0].axhline(1, color='grey', lw=0.8); axes[0].axhspan(0.9, 1.1, color='#f1f3f5', zorder=0); axes[0].set_yscale('log')
axes[1].axhline(0, color='grey', lw=0.8); axes[1].axhspan(-0.3, 0.3, color='#f1f3f5', zorder=0)
axes[0].set_ylabel(rf'$\delta_F(Q^2={Q2_REFERENCE})$ / reference')
axes[1].set_ylabel(rf'shift of $\mathbb{{E}}[F_A]$ at $Q^2={Q2_REFERENCE}$  [$\sigma_{{\rm ref}}$]')
for ax in axes:
    ax.set_xticks(xpos, [f'{v}\n{VARIANT_READS[v]}\n({NBINS[v]} bins)' for v in ladder], fontsize=7)
    ax.grid(axis='y', alpha=0.3, which='both')
axes[0].legend(fontsize=7, ncol=2)
fig.suptitle(f'{FAMILY}: the p_n-information ladder against {REFERENCE_LABEL}', fontsize=10)
fig.tight_layout(); fig.savefig(FIG_DIR / 'headline_ladder.pdf'); plt.show()

In [ ]:
for fit in FITS:
    available = [v for v in [REF, *VARIANTS] if fit in BANDS[v]]
    if not available:
        continue
    fig = plt.figure(figsize=(14, 5))
    grid = fig.add_gridspec(2, 2, height_ratios=[3, 1.2], hspace=0.05, wspace=0.25)
    ax_frac = fig.add_subplot(grid[:, 0]); ax_fa = fig.add_subplot(grid[0, 1]); ax_ratio = fig.add_subplot(grid[1, 1], sharex=ax_fa)
    ref_median = BANDS[REF][fit]['median'] if fit in BANDS[REF] else None
    for v in available:
        band = BANDS[v][fit]
        ax_frac.plot(Q2, 100 * band['frac'], color=VARIANT_COLOR[v], lw=1.8 if v == REF else 1.3, label=VARIANT_LABEL[v])
        ax_fa.plot(Q2, -band['median'], color=VARIANT_COLOR[v], lw=1.3)
        ax_fa.fill_between(Q2, -band['q84'], -band['q16'], color=VARIANT_COLOR[v], alpha=0.15, lw=0)
        if ref_median is not None:
            with np.errstate(divide='ignore', invalid='ignore'):
                ax_ratio.plot(Q2, band['median'] / ref_median, color=VARIANT_COLOR[v], lw=1.3)
                ax_ratio.fill_between(Q2, band['q16'] / ref_median, band['q84'] / ref_median, color=VARIANT_COLOR[v], alpha=0.12, lw=0)
    prior = fa_band(RESULTS[available[0]][fit], Q2, use_prior=True)
    ax_frac.plot(Q2, 100 * prior['frac'], color='grey', ls='--', lw=1.2, label='prior')
    ax_frac.axvline(Q2_REFERENCE, color='grey', lw=0.8, ls=':')
    ax_frac.set_xlabel(r'$Q^2$ [GeV$^2$]'); ax_frac.set_ylabel(r'$\delta_F(Q^2)$ [%]'); ax_frac.legend(fontsize=8)
    ax_frac.set_ylim(0, min(ax_frac.get_ylim()[1], 60))
    ax_fa.set_ylabel(r'$-F_A(Q^2)$'); ax_fa.tick_params(labelbottom=False)
    ax_ratio.axhline(1, color='grey', lw=0.8); ax_ratio.set_ylabel('median / ref'); ax_ratio.set_xlabel(r'$Q^2$ [GeV$^2$]')
    fig.suptitle(f'{FAMILY}: {fit}', y=0.98, fontsize=10)
    fig.savefig(FIG_DIR / f'fa_band_{fit}.pdf', bbox_inches='tight'); plt.show(); plt.close(fig)

## 6. Fitted parameters and nuisance pulls

Top: the independent axial parameters ($M_A$, or $a_1\ldots a_{k_{\max}-4}$) per variant, with
the half-width relative to the reference and the shift of the median in units of the reference
half-width. Bottom, for the fits that float nuisances: the posterior median of each pull and its
68 % half-width divided by the half-width of its prior (drawn with the same truncation as the
fit's `restrict`). A ratio near 1 means the data do not constrain the knob. In `q2_all` that is
the expected symptom of the degeneracy: the MEC and FSI knobs that the 2D fit constrains through
the $p_n$ shape return their priors, and $M_A$ pays for it with a wider posterior. The last
figure plots the width ratio of every knob along the ladder.

In [ ]:
def joint_summary(result):
    summary = result['summary'].iloc[result['joint_indices']]
    return pd.DataFrame({'median': summary['posterior_median'], 'half_width': 0.5 * (summary['q84'] - summary['q16'])})


def nuisance_pulls(result):
    # (names, posterior samples, prior samples) of the fitted nuisance pulls, or None
    if 'ma_joint_samples' in result:
        names, post, prior = result['ma_joint_names'][1:], result['ma_joint_samples'][:, 1:], result['ma_joint_prior_samples'][:, 1:]
    elif 'zexp_nuisance_samples' in result:
        n_axial = len(result['joint_indices'])
        names, post, prior = result['zexp_nuisance_names'][n_axial:], result['zexp_nuisance_samples'][:, n_axial:], result['zexp_nuisance_prior_samples'][:, n_axial:]
    else:
        return None
    return list(names), post, prior


rows, pull_rows = [], []
for fit in FITS:
    ref = joint_summary(RESULTS[REF][fit]) if fit in RESULTS[REF] else None
    for v in [REF, *VARIANTS]:
        result = RESULTS[v].get(fit)
        if result is None:
            continue
        summary = joint_summary(result)
        for name, row in summary.iterrows():
            rows.append(dict(fit=fit, parameter=name, variant=v, median=row['median'], half_width=row['half_width'],
                             width_ratio=row['half_width'] / ref.loc[name, 'half_width'] if ref is not None else np.nan,
                             shift_sigma=(row['median'] - ref.loc[name, 'median']) / ref.loc[name, 'half_width'] if ref is not None else np.nan))
        pulls = nuisance_pulls(result)
        if pulls is None:
            continue
        names, post, prior = pulls
        q16, q50, q84 = np.percentile(post, [16, 50, 84], axis=0)
        p16, p84 = np.percentile(prior, [16, 84], axis=0)
        for i, name in enumerate(names):
            pull_rows.append(dict(fit=fit, knob=name.removesuffix(' pull'), variant=v, median=q50[i],
                                  half_width=0.5 * (q84[i] - q16[i]), width_over_prior=(q84[i] - q16[i]) / (p84[i] - p16[i])))
PARAMS = pd.DataFrame(rows)
param_table = PARAMS.pivot(index=['fit', 'parameter'], columns='variant', values=['median', 'half_width', 'width_ratio', 'shift_sigma'])
param_table = param_table.reindex(columns=pd.MultiIndex.from_product([['median', 'half_width', 'width_ratio', 'shift_sigma'], [REF, *VARIANTS]]))
display(param_table.style.format('{:.3f}', na_rep=''))

PULLS = pd.DataFrame(pull_rows)
if len(PULLS):
    pull_table = PULLS.pivot(index=['fit', 'knob'], columns='variant', values=['median', 'width_over_prior'])
    pull_table = pull_table.reindex(columns=pd.MultiIndex.from_product([['median', 'width_over_prior'], [REF, *VARIANTS]]))
    print('nuisance pulls: posterior median, and 68% width over the prior width (1 = unconstrained by the data)')
    display(pull_table.style.format('{:.2f}', na_rep=''))

    fits_with_pulls = [fit for fit in FITS if fit in set(PULLS.fit)]
    knobs = list(dict.fromkeys(PULLS.knob))
    fig, axes = plt.subplots(1, len(fits_with_pulls), figsize=(3.4 * len(fits_with_pulls), 0.45 * len(knobs) + 1.8), sharey=True, squeeze=False)
    y = np.arange(len(knobs))[::-1]
    for ax, fit in zip(axes[0], fits_with_pulls):
        for k, v in enumerate([REF, *VARIANTS]):
            part = PULLS[(PULLS.fit == fit) & (PULLS.variant == v)].set_index('knob').reindex(knobs)
            if part['median'].isna().all():
                continue
            offset = (k - len(VARIANTS) / 2) * 0.15
            ax.errorbar(part['median'], y + offset, xerr=part['half_width'], fmt='o', ms=3.5, color=VARIANT_COLOR[v], capsize=2, lw=1, label=VARIANT_LABEL[v])
        ax.axvline(0, color='grey', lw=0.8); ax.set_title(fit, fontsize=9); ax.set_xlabel('pull'); ax.grid(axis='x', alpha=0.3)
    axes[0][0].set_yticks(y, knobs, fontsize=8); axes[0][0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'nuisance_pulls.pdf'); plt.show()

    # Width over prior along the ladder: which knobs the data stop constraining without the p_n axis.
    ladder = [*VARIANTS, REF]
    xpos = np.arange(len(ladder))
    fig, axes = plt.subplots(1, len(fits_with_pulls), figsize=(3.6 * len(fits_with_pulls), 3.4), sharey=True, squeeze=False)
    for ax, fit in zip(axes[0], fits_with_pulls):
        for k, knob in enumerate(knobs):
            part = PULLS[(PULLS.fit == fit) & (PULLS.knob == knob)].set_index('variant').reindex(ladder)
            ax.plot(xpos, part['width_over_prior'], 'o-', ms=4, lw=1.2, color=plt.get_cmap('tab10')(k % 10), label=knob)
        ax.axhline(1, color='grey', lw=0.8, ls=':'); ax.set_title(fit, fontsize=9)
        ax.set_xticks(xpos, ladder, fontsize=7, rotation=30, ha='right'); ax.grid(axis='y', alpha=0.3)
    axes[0][0].set_ylabel('posterior width / prior width'); axes[0][0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'nuisance_width_ladder.pdf'); plt.show()

## 7. Axial radius $r_A^2$

Definitions of notebook 7: $r_A^2 = 12/M_A^2$ for the dipole fits, the analytic slope of the
$z$-expansion at $Q^2=0$ otherwise, in fm$^2$; with the fraction of samples at $r_A^2<0$.

In [ ]:
HBARC_GEV_FM = 0.1973269804


def dz_dq2_at_zero(t0, t_cut):
    a, b = np.sqrt(t_cut), np.sqrt(t_cut - t0)
    return b / (a * (a + b) ** 2)


def radius_squared(result, use_prior=False):
    samples = result['prior_samples' if use_prior else 'samples']
    prior = result['spec'].prior
    if prior is None:
        return 12.0 / np.asarray(samples[:, 0], dtype=float) ** 2 * HBARC_GEV_FM ** 2
    coefficients = np.atleast_2d(np.asarray(samples, dtype=float))
    t0, t_cut = prior.t0_gev2, prior.t_cut_gev2
    z0 = (np.sqrt(t_cut) - np.sqrt(t_cut - t0)) / (np.sqrt(t_cut) + np.sqrt(t_cut - t0))
    k = np.arange(1, coefficients.shape[1])
    dfa_dq2 = (coefficients[:, 1:] @ (k * z0 ** (k - 1))) * dz_dq2_at_zero(t0, t_cut)
    return -6.0 / prior.fa_q2_zero * dfa_dq2 * HBARC_GEV_FM ** 2


rows = []
for fit in FITS:
    ref = radius_squared(RESULTS[REF][fit]) if fit in RESULTS[REF] else None
    ref_q = np.percentile(ref, [16, 50, 84]) if ref is not None else None
    for v in [REF, *VARIANTS]:
        if fit not in RESULTS[v]:
            continue
        r2 = radius_squared(RESULTS[v][fit])
        q16, med, q84 = np.percentile(r2, [16, 50, 84])
        rows.append(dict(fit=fit, variant=v, **{'median [fm^2]': med, 'half_width': 0.5 * (q84 - q16),
                         'width_ratio': (q84 - q16) / (ref_q[2] - ref_q[0]) if ref_q is not None else np.nan,
                         'shift_sigma': (med - ref_q[1]) / (0.5 * (ref_q[2] - ref_q[0])) if ref_q is not None else np.nan,
                         'negative [%]': 100 * np.mean(r2 < 0)}))
RADIUS = pd.DataFrame(rows)
radius_table = RADIUS.pivot(index='fit', columns='variant', values=['median [fm^2]', 'width_ratio', 'shift_sigma', 'negative [%]'])
radius_table = radius_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['median [fm^2]', 'width_ratio', 'shift_sigma', 'negative [%]'], [REF, *VARIANTS]]))
display(radius_table.style.format('{:.3f}', na_rep=''))

## 8. Profile scans

PROfit's 1D $\Delta\chi^2$ scans per fitted parameter (fit coordinates), overlaid across the
variants for each fit in `PROFILE_SCAN_FITS`. A wider $M_A$ parabola in `q2_all` is the lost
information; a flat scan of a nuisance means the one-dimensional data do not see that knob.

In [ ]:
def read_profile_points(variant, fit):
    matches = sorted(fit_dir(variant, fit).glob('*_v1_PROfile_points.txt'))
    if not matches:
        return None
    rows = []
    for line in matches[0].read_text().splitlines():
        if not line or line.startswith('#'):
            continue
        index, kind, name, value, dchi2, chi2 = line.split()
        rows.append(dict(index=int(index), kind=kind, parameter=name, value=float(value), dchi2=float(dchi2), chi2=float(chi2)))
    return pd.DataFrame(rows)


for scan_fit in PROFILE_SCAN_FITS:
    scans = {v: read_profile_points(v, scan_fit) for v in [REF, *VARIANTS] if scan_fit in FINISHED[v]}
    scans = {v: s for v, s in scans.items() if s is not None and not s.empty}
    if not scans:
        print(f'no profile scans found for {scan_fit!r}')
        continue
    parameters = list(dict.fromkeys(p for s in scans.values() for p in s.parameter))
    ncol = min(4, len(parameters)); nrow = -(-len(parameters) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.4 * nrow), squeeze=False)
    for ax, parameter in zip(axes.ravel(), parameters):
        for v, scan in scans.items():
            part = scan[scan.parameter == parameter].sort_values('value')
            ax.plot(part.value, part.dchi2, 'o-', ms=3, color=VARIANT_COLOR[v], lw=1.8 if v == REF else 1.2, label=VARIANT_LABEL[v])
        ax.axhline(1, color='grey', lw=0.8, ls=':'); ax.axhline(4, color='grey', lw=0.8, ls=':')
        ax.set_xlabel(parameter); ax.set_ylabel(r'$\Delta\chi^2$'); ax.set_ylim(0, min(ax.get_ylim()[1], 30))
    for ax in axes.ravel()[len(parameters):]:
        ax.set_visible(False)
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f'{FAMILY}: {scan_fit} profile scans along the ladder', y=1.0)
    fig.tight_layout(); fig.savefig(FIG_DIR / f'profile_scans_{scan_fit}.pdf', bbox_inches='tight'); plt.show(); plt.close(fig)

## 9. Summary

One row per fit and variant with the numbers the criteria above refer to, written to
`tables/oned_study/summary_<family>.csv`, and a per-fit verdict table: the $\delta_F$ ratio of
each rung, the fraction of the 1D-to-2D gap (in $\log\delta_F$) that the purity cut alone closes
(`cut closes [%]`), and the fraction closed by the slices below the cut when the $p_n$-range
fits are available (`slices close [%]`). Low `cut closes` with high `slices close` is the
"slicing, not purity" result.

In [ ]:
SUMMARY = (FA[FA.variant != REF]
           .merge(CHI2[['variant', 'fit', 'chi2_data_per_ndof', 'chi2_penalty']], on=['variant', 'fit'], how='left')
           .merge(RADIUS[['variant', 'fit', 'shift_sigma', 'negative [%]']].rename(columns={'shift_sigma': 'r2 shift_sigma', 'negative [%]': 'r2 negative [%]'}),
                  on=['variant', 'fit'], how='left'))
if len(PULLS):
    widest = PULLS.groupby(['fit', 'variant'])['width_over_prior'].max().rename('max pull width/prior').reset_index()
    SUMMARY = SUMMARY.merge(widest, on=['fit', 'variant'], how='left')
SUMMARY['shift_ok'] = (SUMMARY['mean shift [sigma_ref]'].abs() < 0.3).where(SUMMARY['mean shift [sigma_ref]'].notna())
SUMMARY.insert(0, 'family', FAMILY)
SUMMARY.insert(1, 'reference', REFERENCE_LABEL)
SUMMARY.insert(3, 'reads as', SUMMARY['variant'].map(VARIANT_READS))
SUMMARY = SUMMARY.set_index(['fit', 'variant']).reindex(pd.MultiIndex.from_product([FITS, VARIANTS], names=['fit', 'variant'])).dropna(how='all')
display(SUMMARY.style.format({c: '{:.3f}' for c in SUMMARY.columns if SUMMARY[c].dtype.kind == 'f'}, na_rep=''))
SUMMARY.to_csv(TABLE_DIR / f'summary_{FAMILY}.csv', float_format='%.5g')
print(f'wrote {(TABLE_DIR / f"summary_{FAMILY}.csv").relative_to(REPO)}')

# Verdict per fit: how much of the log-gap between q2_all and the reference each rung closes.
ratio = FA.pivot(index='fit', columns='variant', values='ratio to reference').reindex(index=FITS)
VERDICT = pd.DataFrame(index=FITS)
for v in VARIANTS:
    VERDICT[f'{v} / ref'] = ratio[v] if v in ratio else np.nan
if 'q2_all' in ratio:
    gap = np.log(ratio['q2_all'])
    with np.errstate(divide='ignore', invalid='ignore'):
        if 'q2_lowpn' in ratio:
            VERDICT['cut closes [%]'] = 100 * (1 - np.log(ratio['q2_lowpn']) / gap)
        if 'lowpn' in ratio:
            VERDICT['slices close [%]'] = 100 * (1 - np.log(ratio['lowpn']) / gap)
        if 'lowpn_ctrl' in ratio:
            VERDICT['slices+ctrl close [%]'] = 100 * (1 - np.log(ratio['lowpn_ctrl']) / gap)
    VERDICT.loc[gap.abs() < np.log(1.05), [c for c in VERDICT.columns if 'close' in c]] = np.nan   # no gap to close
VERDICT.insert(0, 'family', FAMILY)
display(VERDICT.style.format({c: '{:.3f}' for c in VERDICT.columns if '/ ref' in c} | {c: '{:.0f}' for c in VERDICT.columns if 'close' in c}, na_rep=''))
VERDICT.to_csv(TABLE_DIR / f'verdict_{FAMILY}.csv', float_format='%.5g')
print(f'wrote {(TABLE_DIR / f"verdict_{FAMILY}.csv").relative_to(REPO)}')

## 10. Appendix figure: the $p_n$-information ladder

The compact justification of the two-dimensional grid and of keeping every $p_n$ for the paper appendix, written to `figs/pn_information_ladder.pdf` at one column width. Both panels walk the ladder of section 5 from no $p_n$ information to the full `count50` grid for `APPENDIX_FITS`, loading the chains of both families directly (independent of `FAMILY`). Top: $\delta_F(Q^2_{\rm ref})$ relative to the full grid on Asimov fake data, the information cost of each restriction (MCMC errors from the batch-means ESS, propagated through the ratio). Bottom: the shift of the posterior mean of $F_A(Q^2_{\rm ref})$ on NuWro fake data in units of the full-grid width, the generator dependence each restriction introduces (error from the ESS of both chains).

Reads as: the purity cut alone costs precision and does not remove the generator dependence, the slices recover most of the precision, and the high-$p_n$ control bin removes the shift; only the full grid does both.

In [ ]:
from postfit_physical_parameters import COLUMN_WIDTH, FA_SOURCE_COLORS, column_font_rc

APPENDIX_FITS = [
    ('ma', r'$M_A$, GENIE prior', FA_SOURCE_COLORS['ma'], '-', 'o'),
    ('ma_uniform', r'$M_A$, uniform prior', FA_SOURCE_COLORS['ma'], (0, (4, 2)), 's'),
    ('minerva_k6', r'$z$-exp $k_{\max}=6$, MINERvA prior', FA_SOURCE_COLORS['minerva_k6'], '-', 'D'),
]
APPENDIX_LADDER = ['q2_all', 'q2_lowpn', 'lowpn', 'lowpn_ctrl', REF]
APPENDIX_LABELS = ['$Q^2$ only', '$Q^2$ only,\n$p_n<0.3$ GeV', '$p_n$ slices\n$<0.3$ GeV',
                   'slices +\ncontrol bin', 'full grid']
APPENDIX_TREES = {'q2_all': 'oned_study_fit_results', 'q2_lowpn': 'oned_study_fit_results',
                  'lowpn': 'lowpn_study_fit_results', 'lowpn_ctrl': 'lowpn_study_fit_results',
                  REF: 'binning_study_fit_results'}
PRECISION_FAMILY, SHIFT_FAMILY = 'asimov', 'nuwro'
APPENDIX_FIGURE = REPO / 'figs' / 'pn_information_ladder.pdf'


def fa_point_statistics(result, q2):
    """Mean, standard deviation and batch-means ESS of F_A(q2) over the chain."""
    n = len(result['samples'])
    curve = _fa_curves(result, np.array([q2, q2 + 1e-6]), max_samples=n)[:, 0]
    n_batches = 50
    m = n // n_batches
    batch_var = curve[:n_batches * m].reshape(n_batches, m).mean(axis=1).var(ddof=1)
    ess = min(n, n * curve.var() / (m * batch_var)) if batch_var > 0 else n
    return curve.mean(), curve.std(), ess


rows = []
for family in (PRECISION_FAMILY, SHIFT_FAMILY):
    for fit, *_ in APPENDIX_FITS:
        for variant in APPENDIX_LADDER:
            suite = 'count50' if variant == REF else variant
            result = load_fit(SPEC[fit], suite=suite, burn_in=BURN_IN, thin=THIN,
                              data_root=OUTPUT_ROOT / APPENDIX_TREES[variant] / family)
            if result is None:
                print(f'{family}/{variant}/{fit} missing, skipped')
                continue
            mean, sigma, ess = fa_point_statistics(result, Q2_REFERENCE)
            rows.append(dict(family=family, fit=fit, variant=variant, mean=mean, sigma=sigma, ess=ess))
STATS = pd.DataFrame(rows).set_index(['family', 'fit', 'variant'])

rows = []
for fit, *_ in APPENDIX_FITS:
    ref_p = STATS.loc[(PRECISION_FAMILY, fit, REF)]
    ref_s = STATS.loc[(SHIFT_FAMILY, fit, REF)]
    for x, variant in enumerate(APPENDIX_LADDER):
        if (PRECISION_FAMILY, fit, variant) not in STATS.index or (SHIFT_FAMILY, fit, variant) not in STATS.index:
            continue
        p, s = STATS.loc[(PRECISION_FAMILY, fit, variant)], STATS.loc[(SHIFT_FAMILY, fit, variant)]
        # delta_F = sigma/|mean|; the relative MCMC error of a standard deviation is 1/sqrt(2 ESS)
        ratio = (p.sigma / abs(p['mean'])) / (ref_p.sigma / abs(ref_p['mean']))
        ratio_err = 0.0 if variant == REF else ratio * np.hypot(1 / np.sqrt(2 * p.ess), 1 / np.sqrt(2 * ref_p.ess))
        shift = (s['mean'] - ref_s['mean']) / ref_s.sigma
        shift_err = 0.0 if variant == REF else np.hypot(s.sigma / np.sqrt(s.ess), ref_s.sigma / np.sqrt(ref_s.ess)) / ref_s.sigma
        rows.append(dict(fit=fit, variant=variant, x=x, ratio=ratio, ratio_err=ratio_err,
                         shift=shift, shift_err=shift_err))
APPENDIX_LADDER_TABLE = pd.DataFrame(rows)
display(APPENDIX_LADDER_TABLE.pivot(index='fit', columns='variant', values='ratio')[APPENDIX_LADDER].style.format('{:.3f}'))
display(APPENDIX_LADDER_TABLE.pivot(index='fit', columns='variant', values='shift')[APPENDIX_LADDER].style.format('{:+.2f}'))

with mpl.rc_context({**PUBLICATION_RC, **column_font_rc(COLUMN_WIDTH)}):
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(COLUMN_WIDTH, 4.6), sharex=True,
                                      gridspec_kw=dict(hspace=0.08, height_ratios=[1.1, 1]))
    top.axhline(1, color='#9AA4B2', lw=0.8, zorder=0)
    bottom.axhline(0, color='#9AA4B2', lw=0.8, zorder=0)
    for fit, label, color, linestyle, marker in APPENDIX_FITS:
        part = APPENDIX_LADDER_TABLE[APPENDIX_LADDER_TABLE.fit == fit].sort_values('x')
        style = dict(color=color, ls=linestyle, lw=1.3, marker=marker, ms=3.6,
                     mfc='white' if marker == 's' else color, mew=1.0, capsize=0, zorder=3)
        top.errorbar(part.x, part.ratio, yerr=part.ratio_err, label=label, **style)
        bottom.errorbar(part.x, part['shift'], yerr=part['shift_err'], **style)
    top.set_ylabel(r'$\delta_F$ / $\delta_F$(full grid)')
    top.legend(loc='upper right', handlelength=2.4, borderaxespad=0.4)
    low, high = APPENDIX_LADDER_TABLE.ratio.min(), APPENDIX_LADDER_TABLE.ratio.max()
    top.set_ylim(low - 0.12 * (high - low), high + 0.7 * (high - low))
    bottom.set_ylabel(r'best-fit shift [$\sigma_{\rm full\ grid}$]')
    bottom.set_xticks(range(len(APPENDIX_LADDER)))
    bottom.set_xticklabels(APPENDIX_LABELS)
    bottom.tick_params(axis='x', length=0)
    fig.savefig(APPENDIX_FIGURE, bbox_inches='tight')
    plt.show()
print(f'wrote {APPENDIX_FIGURE}')